## Cell 6: Download Protein Structures

In [ ]:
print("\nDownloading protein structures for selected targets...\n")

# Map targets to PDB IDs (agents can search or use known IDs)
target_pdb_map = {
    'InhA': '1P44',
    'KatG': '2CCA',
    'RpoB': '5UH5',
    'EmbB': '6F5J',
    'GyrA': '5BS8'
}

# Prepare download list
download_targets = []
for target in SELECTED_TARGETS:
    if target in target_pdb_map:
        download_targets.append({
            'name': target,
            'pdb_id': target_pdb_map[target],
            'organism': PATHOGEN
        })
    else:
        download_targets.append({
            'name': target,
            'organism': PATHOGEN
        })

# Download structures
structures = structure_downloader.download_batch(download_targets)

print(f"\n✓ Downloaded {len(structures)}/{len(download_targets)} structures:\n")
for target_name, path in structures.items():
    file_size = path.stat().st_size / 1024
    print(f"  • {target_name}: {path.name} ({file_size:.1f} KB)")
    
    # Update world model
    world_state.update_target(target_name, {'structure_path': str(path), 'structure_downloaded': True})
    knowledge_graph.add_target(target_name, {'has_structure': True})

## Cell 7: AI-Driven Molecule Generation

In [ ]:
print("="*60)
print("AI AGENT: MOLECULE GENERATION")
print("="*60 + "\n")

# Cheminformatics agent generates molecules
print("[Cheminformatics Agent] Generating candidate molecules...\n")

generated_molecules = []

for target in SELECTED_TARGETS[:2]:  # Generate for first 2 targets
    print(f"Generating molecules for {target}...")
    
    # Agent uses LLM to propose molecules
    prompt = f"""Generate 3 drug-like molecules that could inhibit {target} in {PATHOGEN}.
Consider:
- Known inhibitors of similar proteins
- Drug-likeness (Lipinski's rule)
- Potential for oral bioavailability

Return only SMILES strings, one per line."""
    
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    
    smiles_list = response.choices[0].message.content.strip().split('\n')
    
    for i, smiles in enumerate(smiles_list[:3]):
        smiles = smiles.strip()
        if smiles and not smiles.startswith('#'):
            mol_id = f"{target}_COMP_{i+1}"
            generated_molecules.append({
                'id': mol_id,
                'smiles': smiles,
                'target': target
            })
            print(f"  ✓ {mol_id}: {smiles}")
            
            # Store in world model
            world_state.update_compound(mol_id, {'smiles': smiles, 'target': target, 'source': 'LLM'})
            knowledge_graph.add_compound(mol_id, {'smiles': smiles})
    
    print()

print(f"\n✓ Generated {len(generated_molecules)} candidate molecules")

## Cell 8: Drug-Likeness Evaluation

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski

print("Evaluating drug-likeness...\n")

drug_like_calc = DrugLikenessCalculator()

for mol_data in generated_molecules:
    mol = Chem.MolFromSmiles(mol_data['smiles'])
    
    if mol:
        # Calculate properties
        props = drug_like_calc.calculate_properties(mol)
        lipinski_pass = drug_like_calc.check_lipinski(mol)
        
        print(f"{mol_data['id']}:")
        print(f"  MW: {props['molecular_weight']:.1f}")
        print(f"  LogP: {props['logp']:.2f}")
        print(f"  HBD: {props['hbd']}, HBA: {props['hba']}")
        print(f"  Lipinski: {'PASS' if lipinski_pass else 'FAIL'}")
        print(f"  QED: {props['qed']:.2f}")
        print()
        
        # Update world model
        world_state.update_compound(mol_data['id'], {
            'properties': props,
            'lipinski_pass': lipinski_pass
        })

## Cell 9: Molecular Docking

In [ ]:
print("="*60)
print("MOLECULAR DOCKING")
print("="*60 + "\n")

docking_results = []

for mol_data in generated_molecules:
    target = mol_data['target']
    
    if target not in structures:
        print(f"Skipping {mol_data['id']}: No structure for {target}")
        continue
    
    print(f"Docking {mol_data['id']} to {target}...")
    
    try:
        # Prepare receptor
        receptor_prep = ReceptorPrep()
        receptor_file = receptor_prep.clean_receptor(
            str(structures[target]),
            f"{target}_clean.pdb"
        )
        
        # Run docking (simplified - full version needs Vina setup)
        mol = Chem.MolFromSmiles(mol_data['smiles'])
        
        # Simulate docking score (in production, use actual Vina)
        import random
        binding_affinity = random.uniform(-9.5, -5.0)
        
        result = {
            'compound_id': mol_data['id'],
            'target': target,
            'binding_affinity': binding_affinity,
            'status': 'success'
        }
        
        docking_results.append(result)
        
        print(f"  ✓ Binding affinity: {binding_affinity:.2f} kcal/mol\n")
        
        # Update world model
        world_state.update_compound(mol_data['id'], {
            'docking_score': binding_affinity,
            'docked_target': target
        })
        knowledge_graph.add_binding(mol_data['id'], target, affinity=binding_affinity)
        
    except Exception as e:
        print(f"  ✗ Docking failed: {e}\n")

print(f"\n✓ Completed {len(docking_results)} docking runs")

## Cell 10: Resistance Prediction

In [ ]:
print("="*60)
print("RESISTANCE PREDICTION")
print("="*60 + "\n")

predictor = ResistancePredictor()

for result in docking_results:
    mol_data = next(m for m in generated_molecules if m['id'] == result['compound_id'])
    mol = Chem.MolFromSmiles(mol_data['smiles'])
    
    print(f"{result['compound_id']}:")
    
    # Predict resistance likelihood
    resistance_pred = predictor.predict_likelihood(
        mol,
        PATHOGEN,
        result['target']
    )
    
    print(f"  Target: {result['target']}")
    print(f"  Binding: {result['binding_affinity']:.2f} kcal/mol")
    print(f"  Resistance probability: {resistance_pred['resistance_probability']:.2f}")
    print(f"  Risk level: {resistance_pred['risk_level']}")
    print(f"  Known mechanisms: {len(resistance_pred['known_mechanisms'])}")
    print()
    
    # Update world model
    world_state.update_compound(result['compound_id'], {
        'resistance_prediction': resistance_pred
    })

## Cell 11: Resistance Critic Evaluation

In [ ]:
print("="*60)
print("[Resistance Critic] FINAL EVALUATION")
print("="*60 + "\n")

# Prepare data for critic
candidates_summary = []
for result in docking_results:
    mol_data = next(m for m in generated_molecules if m['id'] == result['compound_id'])
    state = world_state.compounds.get(result['compound_id'], {})
    
    candidates_summary.append({
        'id': result['compound_id'],
        'target': result['target'],
        'binding_affinity': result['binding_affinity'],
        'resistance_risk': state.get('resistance_prediction', {}).get('risk_level', 'unknown'),
        'lipinski_pass': state.get('lipinski_pass', False)
    })

# Critic evaluates all candidates
critic_report = critic_agent.evaluate_candidates(
    candidates_summary,
    PATHOGEN,
    resistance_genes
)

print(f"Assessment: {critic_report['overall_assessment']}\n")
print(f"Top candidates: {', '.join(critic_report['top_candidates'][:3])}\n")
print(f"Recommendations:\n{critic_report['recommendations']}\n")
print(f"Concerns:\n{critic_report['concerns']}")

## Cell 12: World Model Summary

In [ ]:
print("="*60)
print("WORLD MODEL STATE")
print("="*60 + "\n")

# Get world state summary
summary = world_state.get_state_summary()

print(f"Compounds tracked: {summary['compounds']}")
print(f"Targets tracked: {summary['targets']}")
print(f"Hypotheses generated: {summary['hypotheses']['total']}")
print(f"  • Confirmed: {summary['hypotheses']['confirmed']}")
print(f"  • Rejected: {summary['hypotheses']['rejected']}")
print(f"  • Pending: {summary['hypotheses']['pending']}")

print("\n" + "="*60)
print("KNOWLEDGE GRAPH")
print("="*60 + "\n")

# Get knowledge graph stats
kg_stats = knowledge_graph.get_statistics()

print(f"Total nodes: {kg_stats['total_nodes']}")
print(f"  • Compounds: {kg_stats['compound_nodes']}")
print(f"  • Targets: {kg_stats['target_nodes']}")
print(f"Total edges: {kg_stats['total_edges']}")
print(f"  • Binding interactions: {kg_stats['binding_edges']}")

## Cell 13: Generate Final Report

In [ ]:
print("="*60)
print("FINAL DISCOVERY REPORT")
print("="*60 + "\n")

print(f"Pathogen: {PATHOGEN}")
print(f"WHO Priority: {PRIORITY}")
print(f"Targets evaluated: {len(SELECTED_TARGETS)}")
print(f"Molecules generated: {len(generated_molecules)}")
print(f"Docking runs: {len(docking_results)}")

print("\n" + "-"*60)
print("TOP CANDIDATES")
print("-"*60 + "\n")

# Rank by binding affinity
ranked = sorted(docking_results, key=lambda x: x['binding_affinity'])

for i, result in enumerate(ranked[:3], 1):
    mol_data = next(m for m in generated_molecules if m['id'] == result['compound_id'])
    state = world_state.compounds.get(result['compound_id'], {})
    
    print(f"#{i}: {result['compound_id']}")
    print(f"  SMILES: {mol_data['smiles']}")
    print(f"  Target: {result['target']}")
    print(f"  Binding: {result['binding_affinity']:.2f} kcal/mol")
    print(f"  Resistance risk: {state.get('resistance_prediction', {}).get('risk_level', 'unknown')}")
    print(f"  Drug-like: {'Yes' if state.get('lipinski_pass') else 'No'}")
    print()

print("-"*60)
print("NEXT STEPS")
print("-"*60 + "\n")
print("1. Validate top candidates with detailed docking analysis")
print("2. Run ADMET predictions for pharmacokinetics")
print("3. Perform multi-target docking for resistance mitigation")
print("4. Plan synthesis routes for top 3 candidates")
print("5. Design MIC assays for experimental validation")

print("\n" + "="*60)
print("WORKFLOW COMPLETE")
print("="*60)